In [1]:
from typing import List, Dict, Any
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.stores import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# 1. Initialize our local execution engines
llm = ChatOllama(model="llama3.2", temperature=0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# 2. A standard Python dict to manage user sessions globally
# Structure: { session_id: [ {"role": "user/assistant", "content": "text"} ] }
session_database = {}

print("✓ Cell 1: Local models and standard Python session dictionary ready.")

✓ Cell 1: Local models and standard Python session dictionary ready.


/tmp/ipykernel_107081/1536184759.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
class SimpleParentChildRetriever:
    def __init__(self, embeddings_engine):
        # FAISS only holds tiny child chunks for precise mathematical vector matching
        # We pass a single initialization token so FAISS can safely map vector dimensions
        init_doc = [Document(page_content="init")]
        self.vector_store = FAISS.from_documents(init_doc, embeddings_engine)
        
        # A simple dictionary store to map parent string IDs to their full un-cut texts
        self.parent_store = {}
        
        # Slicing configurations
        self.parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=0)
        self.child_splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20)

    def ingest_document(self, source_name: str, raw_text: str):
        # Step A: Slice into larger parent blocks and save them to our dict store
        fake_doc = Document(page_content=raw_text)
        parent_chunks = self.parent_splitter.split_documents([fake_doc])
        
        for p_idx, p_chunk in enumerate(parent_chunks):
            parent_id = f"{source_name}_p_{p_idx}"
            self.parent_store[parent_id] = p_chunk.page_content
            
            # Step B: Slice that specific parent into tiny child sentences
            child_chunks = self.child_splitter.split_documents([p_chunk])
            for c_chunk in child_chunks:
                # Stamp the parent's lookup ID directly into the child's metadata
                c_chunk.metadata["parent_id"] = parent_id
                
            # Step C: Save only the tiny child chunks to the Vector DB
            self.vector_store.add_documents(child_chunks)

    def retrieve_context(self, user_query: str) -> str:
        # Step A: Match the top 2 closest child sentences in the Vector DB
        child_matches = self.vector_store.similarity_search(user_query, k=2)
        
        # Step B: Collect unique parent IDs from those child matches
        parent_ids = list(set(c.metadata["parent_id"] for c in child_matches if "parent_id" in c.metadata))
        
        # Step C: Pull the full, un-diluted text blocks from our parent store
        retrieved_blocks = [self.parent_store[p_id] for p_id in parent_ids if p_id in self.parent_store]
        
        # Glue them together into a clean string for the LLM
        return "\n\n".join(retrieved_blocks)

# Instantiate our simple custom retriever
retriever = SimpleParentChildRetriever(embeddings_engine=embeddings)
print("✓ Cell 2: SimpleParentChildRetriever compiled explicitly.")

✓ Cell 2: SimpleParentChildRetriever compiled explicitly.


In [4]:
system_doc = """
MeshQuery is our internal high-scale distributed indexing engine built in 2026. 
It uses a specialized sharding strategy optimized for high-throughput Kafka topics.
Unlike standard relational databases, MeshQuery caches query indexes using H3 spatial indexing clusters 
and routes requests across nodes using an ultra-low latency RSocket protocol layer.
The default port for MeshQuery cluster coordination is 9092.
"""

# Ingest using our custom clean class method
retriever.ingest_document(source_name="meshquery_spec", raw_text=system_doc)

print("✓ Cell 3: Data split and mapped: child chunks stored in FAISS, parents stored in Python dict.")

✓ Cell 3: Data split and mapped: child chunks stored in FAISS, parents stored in Python dict.


In [5]:
def execute_conversational_rag(session_id: str, user_question: str):
    # 1. Fetch relevant parent context using our retriever class
    context_string = retriever.retrieve_context(user_question)
    
    # 2. Look up or initialize the chat history list for this specific user session
    if session_id not in session_database:
        session_database[session_id] = []
    chat_history = session_database[session_id]
    
    # 3. Construct a standard, explicit OpenAI-style message payload array
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert system architecture assistant. Answer the user's question "
                "using ONLY the provided context block below. If the answer cannot be found "
                "explicitly in the context, say 'I do not have that information.'\n\n"
                f"Context:\n{context_string}"
            )
        }
    ]
    
    # Dynamically append past conversational turns into our list payload
    for turn in chat_history:
        messages.append(turn)
        
    # Append the current active user turn
    messages.append({"role": "user", "content": user_question})
    
    # 4. Stream the execution tokens directly from the model
    print(f"\nUser: {user_question}")
    print("AI: ", end="", flush=True)
    
    ai_response_string = ""
    for chunk in llm.stream(messages):  # Passes the raw list of dicts directly!
        print(chunk.content, end="", flush=True)
        ai_response_string += chunk.content
    print("\n")
    
    # 5. Core Concept: Save the completed turn to our history database
    chat_history.append({"role": "user", "content": user_question})
    chat_history.append({"role": "assistant", "content": ai_response_string})

print("✓ Cell 4: Conversational execution loop compiled using native dictionary structures.")

✓ Cell 4: Conversational execution loop compiled using native dictionary structures.


In [6]:
session_key = "abhishek_dev_session"

# Turn 1: Establish explicit context
execute_conversational_rag(session_key, "What is MeshQuery?")

# Turn 2: Ask an ambiguous follow-up relying on memory state tracking
execute_conversational_rag(session_key, "What port does it use?")


User: What is MeshQuery?
AI: MeshQuery is our internal high-scale distributed indexing engine, built in 2026.


User: What port does it use?
AI: The default port for MeshQuery cluster coordination is 9092.

